In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gradte/loan-prediction-problem-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/loan-prediction-problem-dataset


In [2]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_PATH = "/kaggle/input/loan-prediction-problem-dataset/imbalanced_dataset.csv"

In [4]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())

Shape: (472, 13)
    Loan_ID  Gender Married Dependents Education Self_Employed  \
0  LP001316    Male     Yes          0  Graduate            No   
1  LP001387  Female     Yes          0  Graduate           NaN   
2  LP001157  Female      No          0  Graduate            No   
3  LP002297    Male      No          0  Graduate            No   
4  LP002141    Male     Yes         3+  Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             2958             2900.0       131.0             360.0   
1             2929             2333.0       139.0             360.0   
2             3086                0.0       120.0             360.0   
3             2500            20000.0       103.0             360.0   
4             2666             2083.0        95.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0     Semiurban           Y  
1             1.0     Semiurban           Y  
2             1.0    

In [5]:
possible_targets = ["loan_status","approved","Loan_Status","LoanApproved","target","label","Eligible","approved_flag","LoanStatus"]
target_col = None
for t in possible_targets:
    if t in df.columns:
        target_col = t
        break
if target_col is None:
    target_col = df.columns[-1]
    print(f"Using last column as target: {target_col}")
else:
    print(f"Detected target column: {target_col}")

print("Target counts:\n", df[target_col].value_counts(dropna=False))

Detected target column: Loan_Status
Target counts:
 Loan_Status
Y    422
N     50
Name: count, dtype: int64


In [6]:
X = df.drop(columns=[target_col])
y = df[target_col].copy()

In [7]:
if y.dtype.kind in 'O' or y.dtype.name == 'category':
    y = y.astype(str).str.strip().replace({'Y':'1','N':'0','Yes':'1','No':'0','Approved':'1','Denied':'0'})
    try:
        y = pd.to_numeric(y)
    except:
        y = (y != y.mode()[0]).astype(int)


In [8]:
if y.nunique() > 2:
    # convert to binary by treating most frequent as "negative"
    most_freq = y.mode()[0]
    y = (y != most_freq).astype(int)

print("Processed target distribution:\n", y.value_counts())

Processed target distribution:
 Loan_Status
1    422
0     50
Name: count, dtype: int64


In [9]:
numeric_feats = X.select_dtypes(include=[np.number]).columns.tolist()
cat_feats = X.select_dtypes(include=['object','category','bool']).columns.tolist()
print("Numeric:", numeric_feats)
print("Categorical:", cat_feats)

Numeric: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Categorical: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']


In [10]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


In [11]:
low_card = [c for c in cat_feats if X[c].nunique(dropna=False) <= 10]
high_card = [c for c in cat_feats if X[c].nunique(dropna=False) > 10]
transformers = []
if numeric_feats:
    transformers.append(('num', numeric_transformer, numeric_feats))
if low_card:
    transformers.append(('low_card', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), low_card))
if high_card:
    transformers.append(('high_card', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='MISSING')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))]), high_card))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print("Train/test shapes:", X_train.shape, X_test.shape)

Train/test shapes: (377, 12) (95, 12)


In [13]:
try:
    from xgboost import XGBClassifier
    clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=200)
    model_name = "XGBoost"
except Exception as e:
    print("xgboost not available; using sklearn GradientBoosting. Exception:", e)
    from sklearn.ensemble import GradientBoostingClassifier
    clf = GradientBoostingClassifier(random_state=42, n_estimators=200)
    model_name = "GradientBoosting(sklearn)"

pipeline = Pipeline([('pre', preprocessor), ('clf', clf)])

In [14]:
pipeline.fit(X_train, y_train)
print("Trained:", model_name)

Trained: XGBoost


In [15]:
y_pred = pipeline.predict(X_test)
y_proba = None
if hasattr(pipeline.named_steps['clf'], 'predict_proba'):
    y_proba = pipeline.predict_proba(X_test)[:,1]
elif hasattr(pipeline.named_steps['clf'], 'decision_function'):
    from scipy.special import expit
    y_proba = expit(pipeline.decision_function(X_test))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_proba) if (y_proba is not None and len(np.unique(y_test))==2) else None


In [16]:

print("Accuracy: {:.4f} Precision: {:.4f} Recall: {:.4f} F1: {:.4f}".format(acc,prec,rec,f1))
if roc is not None:
    print("ROC AUC: {:.4f}".format(roc))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.8632 Precision: 0.9000 Recall: 0.9529 F1: 0.9257
ROC AUC: 0.5635
              precision    recall  f1-score   support

           0       0.20      0.10      0.13        10
           1       0.90      0.95      0.93        85

    accuracy                           0.86        95
   macro avg       0.55      0.53      0.53        95
weighted avg       0.83      0.86      0.84        95



In [17]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)


Confusion matrix:
 [[ 1  9]
 [ 4 81]]


In [18]:
if hasattr(pipeline.named_steps['clf'], 'feature_importances_'):
    imps = pipeline.named_steps['clf'].feature_importances_
    # attempt to get names from preprocessor
    feat_names = []
    for name, trans, cols in pipeline.named_steps['pre'].transformers_:
        if hasattr(trans, 'named_steps'):
            last = trans.named_steps[list(trans.named_steps.keys())[-1]]
            if hasattr(last, 'get_feature_names_out'):
                names = last.get_feature_names_out(cols)
            else:
                names = cols
        else:
            if hasattr(trans, 'get_feature_names_out'):
                names = trans.get_feature_names_out(cols)
            else:
                names = cols
        if isinstance(names, np.ndarray):
            names = names.tolist()
        feat_names.extend(names)
    if len(feat_names) == len(imps):
        fi = pd.Series(imps, index=feat_names).sort_values(ascending=False).head(30)
    else:
        fi = pd.Series(imps).sort_values(ascending=False).head(30)
    print("Top feature importances:\n", fi.head(20))
else:
    print("Model does not expose feature_importances_")

Top feature importances:
 Credit_History       0.306456
Self_Employed        0.151750
Loan_Amount_Term     0.111697
CoapplicantIncome    0.077600
Education            0.066384
Dependents           0.063565
Married              0.053520
LoanAmount           0.049419
ApplicantIncome      0.042149
Gender               0.040035
Property_Area        0.037426
Loan_ID_LP002362     0.000000
Loan_ID_LP002361     0.000000
Loan_ID_LP002348     0.000000
Loan_ID_LP002305     0.000000
Loan_ID_LP002345     0.000000
Loan_ID_LP002337     0.000000
Loan_ID_LP002332     0.000000
Loan_ID_LP002319     0.000000
Loan_ID_LP002317     0.000000
dtype: float32


In [19]:
joblib.dump(pipeline, "loan_xgb_model.pkl")
print("Saved pipeline to loan_xgb_model.pkl")


Saved pipeline to loan_xgb_model.pkl


In [20]:
sample = X_test.head(10).copy()
sample['pred_label'] = y_pred[:10]
if y_proba is not None:
    sample['pred_proba'] = y_proba[:10]
print("Sample predictions:\n", sample)


Sample predictions:
       Loan_ID  Gender Married Dependents     Education Self_Employed  \
259  LP002753  Female      No          1      Graduate           NaN   
56   LP001422  Female      No          0      Graduate            No   
282  LP002940    Male      No          0  Not Graduate            No   
228  LP002112    Male     Yes          2      Graduate           Yes   
16   LP001744    Male      No          0      Graduate            No   
107  LP001120    Male      No          0      Graduate            No   
315  LP001825    Male     Yes          0      Graduate            No   
265  LP002051    Male     Yes          0      Graduate            No   
229  LP001743    Male     Yes          2      Graduate            No   
186  LP002308    Male     Yes          0  Not Graduate            No   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
259             3652                0.0        95.0             360.0   
56             10408                0.0 

In [21]:
def advisory(prob):
    if prob is None:
        return "No probability available — use predicted label"
    if prob >= 0.70:
        return "Approve — high likelihood"
    elif prob >= 0.40:
        return "Manual review — borderline. Request additional docs (income proof, credit report)"
    else:
        return "Reject — low likelihood. Recommend improve credit history / reduce loan amount"

if y_proba is not None:
    sample['advice'] = sample['pred_proba'].apply(advisory)
    print("Sample with advice:\n", sample[['pred_label','pred_proba','advice']])


Sample with advice:
      pred_label  pred_proba                                             advice
259           1    0.863896                          Approve — high likelihood
56            0    0.471954  Manual review — borderline. Request additional...
282           1    0.978580                          Approve — high likelihood
228           1    0.984910                          Approve — high likelihood
16            1    0.999948                          Approve — high likelihood
107           1    0.994672                          Approve — high likelihood
315           1    0.998844                          Approve — high likelihood
265           1    0.999817                          Approve — high likelihood
229           1    0.997658                          Approve — high likelihood
186           1    0.999826                          Approve — high likelihood
